<a href="https://colab.research.google.com/github/kamilabalabatyr/Data-Science/blob/main/hw6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашнее задание 6: продвинутый Numpy

**Как устроено.** Две части по примерно 45 минут: **A** — функции и трансляция (урок 11), **B** — линейная алгебра и случайные числа (урок 12). После каждого пункта — ✅ проверка: запустите её.
- ★ — по образцу из урока, ★★ — соединить две идеи, ★★★ — подумать самому (по желанию).
- Ответы записывайте в переменные с указанными именами — проверки ищут именно их.
- 🟢 **Обязательно:** A1–A7 и B1–B8. Отмеченное «по желанию» и бонус C — для тех, кто хочет глубже.
- **Без циклов `for`** во всём задании, кроме мест, где это сказано явно.

**Окна рядом.** У каждого пункта указан слайд урока (🖥, номер — в правом нижнем углу слайда) и шаг рабочей тетради `lesson6_workbook.ipynb` (📓), где та же идея разобрана подробно. Застряли — откройте их.

**Не только «напиши по условию».** Здесь есть: 🔍 найди ошибку в чужом коде, 🧱 граничные случаи, 🧹 грязные данные, 🔄 требования поменялись, 🗣 ответьте своими словами «почему».

**Сдаётся этот файл.** Перед сдачей: «Restart» → «Run All» — всё должно выполниться сверху вниз, а проверки — дать ✅.

---
# Часть A. Фитнес-трекер: функции и трансляция

Пятеро друзей 4 недели носили фитнес-браслеты. Строки — 28 дней (пн–вс, пн–вс…), столбцы — друзья. Данные «как есть»: со сбоями и пропусками.

In [ ]:
import numpy as np

friends = np.array(["Аня", "Борис", "Вика", "Гоша", "Дана"])
steps = np.random.default_rng(11).normal(8500, 3000, size=(28, 5)).round()
# 🧹 настоящие данные браслетов — с ошибками:
steps[4, 1] = -1200          # сбой: отрицательные шаги
steps[10, 3] = 99999         # сбой: браслет «сошёл с ума»
steps[[6, 20], 0] = np.nan   # Аня забыла браслет дома
print(steps.shape)
print(steps[:5])

(28, 5)
[[ 8603. 12579. 12174.  6969.  7606.]
 [ 6918. 10209.  8332. 10741.  2958.]
 [13200.  8211. 10541.  8090.  7363.]
 [ 9889. 10974.  7892.  8042. 10557.]
 [ 5889. -1200.  9685.  6488.  2739.]]


## A1 ★ 🧹 Чистка данных · 🖥 слайды 5–6 · 📓 шаги 2–3
1. `steps_clean` — копия данных, где **отрицательные** значения заменены на 0, а значения **больше 50 000** — на `np.nan`. Исходный `steps` не меняйте: сырые данные храним как есть.
2. `n_missing` — сколько пропусков (`nan`) у каждого друга **после** чистки.

In [ ]:
steps_clean = np.clip(steps, 0, None)
steps_clean = np.where(steps_clean>50000, np.nan, steps_clean)

n_missing = np.isnan(steps_clean).sum(axis=0)

In [ ]:
# ✅ Проверка A1 — просто запустите ячейку
try:
    assert steps[4, 1] == -1200 and steps[10, 3] == 99999, "исходный steps изменился — работайте с копией (np.clip и np.where возвращают новый массив)"
    assert steps_clean.shape == (28, 5)
    assert steps_clean[4, 1] == 0, "отрицательные шаги → 0"
    assert np.isnan(steps_clean[10, 3]), "99 999 → nan"
    assert np.nanmin(steps_clean) >= 0 and np.nanmax(steps_clean) <= 50_000
    assert list(n_missing) == [2, 0, 0, 1, 0], f"n_missing: {n_missing}"
    print("✅ A1: всё верно!")
except AssertionError as e:
    print("❌ A1: пока не так —", e)
except Exception as e:
    print(f"❌ A1: ошибка {type(e).__name__}: {e}")

✅ A1: всё верно!


🗣 **Почему так?** Отрицательные шаги мы заменили нулём, а 99 999 — пропуском, а не числом 50 000. Почему так честнее? Что стало бы со средним Гоши, если бы записали 50 000?

**Ваш ответ:** 99 999 — сбой браслета, настоящее число шагов в тот день неизвестно. Если записать 50 000, это выдуманное значение, которое в несколько раз больше обычных ~8 000 и сильно завысит среднее Гоши. А nan честно означает «нет данных», и nan-функции этот день просто пропустят.

## A2 ★ Статистика без пропусков · 🖥 слайды 7–8 · 📓 шаги 5–6
По каждому другу (по столбцам), **не учитывая `nan`**:
1. `mean_steps` — среднее;
2. `median_steps` — медиана;
3. `p90` — 90-й перцентиль.

Подсказка: у функций есть версии с `nan` в начале имени.

In [ ]:
mean_steps = np.nanmean(steps_clean, axis=0)
median_steps = np.nanmedian(steps_clean, axis=0)
p90 = np.nanpercentile(steps_clean, 90, axis=0)

In [ ]:
# ✅ Проверка A2 — просто запустите ячейку
try:
    assert np.shape(mean_steps) == (5,), "по одному числу на друга: axis=0"
    assert not np.isnan(mean_steps).any(), "в ответе есть nan — нужна версия функции, которая пропускает nan"
    assert np.allclose(mean_steps, [8431.0, 8422.9, 9503.1, 8180.2, 8444.4], atol=0.1), f"mean_steps: {mean_steps.round(1)}"
    assert np.allclose(median_steps, [8176.0, 9452.5, 9554.5, 8221.0, 8191.5]), f"median_steps: {median_steps}"
    assert np.allclose(p90, [12928.5, 12330.5, 12135.5, 10482.0, 12317.1], atol=0.1), f"p90: {p90.round(1)}"
    print("✅ A2: всё верно!")
except AssertionError as e:
    print("❌ A2: пока не так —", e)
except Exception as e:
    print(f"❌ A2: ошибка {type(e).__name__}: {e}")

✅ A2: всё верно!


:🗣 **Почему так?** Посчитайте среднее и медиану Гоши (столбец 3) по **сырым** данным `steps` (с `nanmean` / `nanmedian`) и сравните с чистыми. Что изменилось сильнее — и почему?

In [ ]:
raw_mean_steps = np.nanmean(steps[:,3])
raw_median_steps = np.nanmedian(steps[:,3])

clean_mean_steps = np.nanmean(steps_clean[:,3])
clean_median_steps = np.nanmedian(steps_clean[:,3])


print(f"Statistics for raw data 'steps': mean: {raw_mean_steps:.2f}, median: {raw_median_steps}")
print(f"Statistics for clear data 'steps': mean: {clean_mean_steps:.2f}, median: {clean_median_steps}")

Statistics for raw data 'steps': mean: 11459.43, median: 8333.0
Statistics for clear data 'steps': mean: 8180.19, median: 8221.0


**Ваш ответ:** По чистым данным мы получаем максимально правдоподобные статистики. У сырых дата есть дни/шаги где отклонения не естественно высокие/низкие, за сяет этого там более высокие значения, потому что среднее считается через сумму всех дней, и одно огромное число 99 999 сильно её увеличивает. Значит, среднее чувствительно к выбросам, а медиана устойчива.

## A3 ★ Километры · 🖥 слайд 5 · 📓 шаг 2
Длина шага — 0.75 м.
1. `km` — километры каждого друга в каждый день, **округлённые до 1 знака** (форма `(28, 5)`, пропуски остаются пропусками).
2. `total_km` — сколько километров каждый прошёл за 4 недели (без пропусков).
3. `record_day` — номер дня, когда у каждого был рекорд шагов. Обычный `argmax` споткнётся о `nan` — найдите в документации подходящую функцию.

In [ ]:
km = np.round(((steps_clean*0.75)/1000), 1)
total_km = np.nansum(km, axis=0)
record_day = np.nanargmax(steps_clean, axis=0)

In [ ]:
# ✅ Проверка A3 — просто запустите ячейку
try:
    assert np.shape(km) == (28, 5)
    assert np.isnan(km[6, 0]), "пропуск в шагах — пропуск и в километрах"
    assert np.allclose(np.nan_to_num(km), np.nan_to_num(km).round(1)), "km округлите до 1 знака"
    assert np.allclose(total_km, [164.4, 176.9, 199.5, 165.6, 177.2], atol=0.3), f"total_km: {np.round(total_km, 1)}"
    assert list(record_day) == [18, 0, 23, 27, 18], f"record_day: {record_day}"
    print("✅ A3: всё верно!")
except AssertionError as e:
    print("❌ A3: пока не так —", e)
except Exception as e:
    print(f"❌ A3: ошибка {type(e).__name__}: {e}")

✅ A3: всё верно!


## A4 ★★ По неделям · 🖥 слайды 6, 9 · 📓 шаги 4, 7
🔮 Сначала предскажите: какая форма у `steps_clean.reshape(4, 7, 5)` и что означает каждая ось?
1. `weekly` — сумма шагов каждого друга за каждую неделю, форма `(4, 5)` (без пропусков).
2. `growth` — изменение к предыдущей неделе, форма `(3, 5)`.
3. `best_week` — номер лучшей недели каждого друга.

In [ ]:
weekly = np.nansum(steps_clean.reshape(4, 7, 5), axis=1)
growth = np.diff(weekly, axis=0)
best_week = np.argmax(weekly, axis=0)

In [ ]:
# ✅ Проверка A4 — просто запустите ячейку
try:
    assert np.shape(weekly) == (4, 5), f"weekly: форма {np.shape(weekly)}. Какая ось reshape(4, 7, 5) — дни недели?"
    assert np.allclose(weekly[0], [50557, 56871, 59813, 54008, 50485]), f"первая неделя: {weekly[0]}"
    assert np.shape(growth) == (3, 5), "diff по неделям: axis=0"
    assert growth[0, 3] == -9781, f"Гоша, неделя 2 к неделе 1: {growth[0, 3]}"
    assert list(best_week) == [2, 1, 3, 2, 2], f"best_week: {best_week}"
    print("✅ A4: всё верно!")
except AssertionError as e:
    print("❌ A4: пока не так —", e)
except Exception as e:
    print(f"❌ A4: ошибка {type(e).__name__}: {e}")

✅ A4: всё верно!


## A5 ★★ Личные цели · 🖥 слайд 10 · 📓 шаг 8
У каждого своя цель шагов в день — `goals`.
1. `achieved` — сколько дней каждый выполнил свою цель.
2. `pct` — процент выполнения цели каждым в каждый день, форма `(28, 5)`.
3. `mean_pct` — средний процент выполнения у каждого (без пропусков), округлённый до 1 знака.

In [ ]:
goals = np.array([8000, 10000, 6000, 9000, 7000])
achieved = (steps_clean>=goals).sum(axis=0)
pct = ((steps_clean/goals)*100)
mean_pct = np.round(np.nanmean(pct, axis=0), 1)

In [ ]:
# ✅ Проверка A5 — просто запустите ячейку
try:
    assert list(achieved) == [13, 13, 25, 10, 20], f"achieved: {achieved}"
    assert np.shape(pct) == (28, 5)
    assert np.allclose(mean_pct, [105.4, 84.2, 158.4, 90.9, 120.6]), f"mean_pct: {mean_pct}"
    print("✅ A5: всё верно!")
except AssertionError as e:
    print("❌ A5: пока не так —", e)
except Exception as e:
    print(f"❌ A5: ошибка {type(e).__name__}: {e}")

✅ A5: всё верно!


🧱 **Граничный случай.** Чему равно `np.nan >= 8000`? Засчитываются ли дни без браслета как выполненные? Правильно ли это?

**Ваш ответ:** np.nan >= 8000 даёт False, поэтому дни без браслета считаются как невыполненные, хотя на самом деле результат неизвестен. Это занижает показатель Ани. Честнее считать долю выполненных дней среди дней с данными, а заполнять пропуски похожим днём можно, но это уже допущение.

## A6 ★ 🔍 Найди ошибку · 🖥 слайд 11 · 📓 шаг 9
Коллега хотел узнать, насколько каждый друг отличался от **среднего по компании в этот день**: из каждой строки вычесть её среднее. Получил ошибку. Почините — результат `dev` должен иметь форму `(28, 5)`.

In [ ]:
try:
    day_mean = np.nanmean(steps_clean, axis=1, keepdims=True)
    dev = steps_clean - day_mean
except ValueError as e:
    print("ValueError:", e)

In [ ]:
# ✅ Проверка A6 — просто запустите ячейку
try:
    assert np.shape(dev) == (28, 5), f"dev: форма {np.shape(dev)}"
    assert np.allclose(np.nanmean(dev, axis=1), 0), "в каждой строке среднее отклонение должно быть 0"
    print("✅ A6: всё верно!")
except AssertionError as e:
    print("❌ A6: пока не так —", e)
except Exception as e:
    print(f"❌ A6: ошибка {type(e).__name__}: {e}")

✅ A6: всё верно!


🗣 **Почему так?** Представьте, что друзей 28 и таблица квадратная `(28, 28)`. Что сделал бы исходный код коллеги — упал бы или нет? Чем это опасно?

**Ваш ответ:** При квадратной таблице (28, 28) код коллеги не упал бы: формы (28, 28) и (28,) совместимы, потому что справа 28 и 28 равны. Но (28,) растягивается как строка, поэтому из каждого столбца вычиталось бы чужое среднее: из столбца друга j — среднее дня j. Результат получился бы неправильным, а ошибки не было бы. Это опасно, потому что такую «тихую» ошибку легко не заметить. Защита — писать keepdims=True и проверять смысл результата (например, что среднее отклонение в каждой строке ≈ 0).

## A7 ★★ 🔄 Требования поменялись · 🖥 слайд 11 · 📓 шаг 9
Тренер решил: в выходные (6-й и 7-й день каждой недели) цель **на 20% ниже**.
1. `day_factor` — массив из 28 чисел: 1 в будни, 0.8 в выходные. Подсказка: `np.tile`.
2. `goal_table` — цель каждого друга в каждый день, форма `(28, 5)`.
3. `achieved2` — сколько дней каждый выполнил цель по новым правилам.

In [ ]:
day_factor = np.tile([1,1,1,1,1,0.8,0.8],4)
goal_table = day_factor[:, None]*goals
achieved2 = (steps_clean>=goal_table).sum(axis=0)

In [ ]:
# ✅ Проверка A7 — просто запустите ячейку
try:
    assert np.shape(day_factor) == (28,)
    assert np.shape(goal_table) == (28, 5), f"goal_table: форма {np.shape(goal_table)}"
    assert np.allclose(goal_table[5], [6400, 8000, 4800, 7200, 5600]), "суббота первой недели: цели × 0.8"
    assert np.allclose(goal_table[7], goals), "понедельник второй недели: обычные цели"
    assert list(achieved2) == [14, 14, 26, 13, 21], f"achieved2: {achieved2}"
    print("✅ A7: всё верно!")
except AssertionError as e:
    print("❌ A7: пока не так —", e)
except Exception as e:
    print(f"❌ A7: ошибка {type(e).__name__}: {e}")

✅ A7: всё верно!


## A8 ★★★ (по желанию) Похожий друг · 🖥 слайд 12 · 📓 шаги 10, L4
Опишем каждого друга четырьмя числами — средними шагами по неделям — и найдём, кто на кого больше похож.
1. `profile` — средние шаги за каждую неделю, форма `(5, 4)`: друзья × недели (без пропусков).
2. `dist` — расстояния между всеми парами друзей, форма `(5, 5)`, через broadcasting. На диагонали (друг сам с собой) запишите `np.inf` — найдите `np.fill_diagonal`.
3. `buddy` — номер самого похожего друга для каждого.

In [ ]:
profile = np.nanmean(steps_clean.reshape(4, 7, 5), axis=1).T
dist = np.linalg.norm(profile[:, None, :] - profile[None, :, :], axis=2)
np.fill_diagonal(dist, np.inf)
buddy = dist.argmin(axis=1)

In [ ]:
# ✅ Проверка A8 — просто запустите ячейку
try:
    assert np.shape(profile) == (5, 4), f"profile: форма {np.shape(profile)}"
    assert np.shape(dist) == (5, 5) and np.isinf(np.diag(dist)).all()
    assert np.allclose(dist, dist.T), "расстояние от A до B равно расстоянию от B до A"
    assert round(dist[0, 3]) == 834, f"Аня — Гоша: {dist[0, 3]:.0f}"
    assert list(buddy) == [3, 0, 1, 0, 3], f"buddy: {buddy}"
    print("✅ A8: всё верно!")
except AssertionError as e:
    print("❌ A8: пока не так —", e)
except Exception as e:
    print(f"❌ A8: ошибка {type(e).__name__}: {e}")

✅ A8: всё верно!


🔗 Это ровно то, что делает алгоритм **KNN** из курса машинного обучения: описать объекты числами, посчитать расстояния и взять ближайших соседей.

---
# Часть B. Пекарня: линейная алгебра и случайные числа

Пекарня печёт 4 вида изделий. Рецепт: сколько каждого ингредиента уходит на **одно** изделие. Заказы — с понедельника по субботу.

In [ ]:
products = np.array(["хлеб", "батон", "круассан", "пирожок"])
ingredients = np.array(["мука, г", "масло, г", "яйца, шт", "сахар, г"])

recipe = np.array([          # строки — изделия, столбцы — ингредиенты
    [500, 10, 0, 10],        # хлеб
    [400, 20, 1, 20],        # батон
    [ 60, 30, 1, 10],        # круассан
    [ 80, 10, 1, 15],        # пирожок
])
orders = np.array([          # строки — дни (пн–сб), столбцы — изделия
    [40, 25, 30, 50],
    [38, 22, 28, 45],
    [42, 27, 35, 55],
    [45, 30, 32, 60],
    [55, 35, 45, 70],
    [60, 40, 55, 80],
])
price = np.array([450, 400, 350, 250])        # цена продажи, ₸
ing_price = np.array([0.25, 3.5, 60, 0.5])    # ₸ за г муки, г масла, яйцо, г сахара
print(recipe.shape, orders.shape)

(4, 4) (6, 4)


## B1 ★ Расход ингредиентов · 🖥 слайды 16–17 · 📓 шаги L1–L2
1. `need` — сколько каждого ингредиента нужно в каждый день, форма `(6, 4)`.
2. `week_need` — за всю неделю.
3. `flour_kg` — муки за неделю в килограммах.

In [ ]:
need = orders @ recipe              # (дни × изделия) @ (изделия × ингредиенты) = (6, 4)
week_need = need.sum(axis=0)        # сумма по дням
flour_kg = week_need[0] / 1000      # мука — ингредиент 0, граммы → кг

In [ ]:
# ✅ Проверка B1 — просто запустите ячейку
try:
    assert np.shape(need) == (6, 4), f"need: форма {np.shape(need)}"
    assert list(need[0]) == [35800, 2300, 105, 1950], f"понедельник: {need[0]}"
    assert list(week_need) == [253900, 16730, 764, 14030], f"week_need: {week_need}"
    assert np.isclose(flour_kg, 253.9), f"flour_kg: {flour_kg}"
    print("✅ B1: всё верно!")
except AssertionError as e:
    print("❌ B1: пока не так —", e)
except Exception as e:
    print(f"❌ B1: ошибка {type(e).__name__}: {e}")

✅ B1: всё верно!


## B2 ★★ Себестоимость и прибыль · 🖥 слайд 17 · 📓 шаг L2
1. `unit_cost` — себестоимость **одного** изделия каждого вида.
2. `margin` — прибыль с одного изделия (цена − себестоимость).
3. `profit_day` — прибыль пекарни в каждый день.
4. `best_product` — **название** изделия, которое принесло больше всего прибыли за неделю.

In [ ]:
unit_cost = recipe @ ing_price                              # ингредиенты изделия × их цены, сложить
margin = price - unit_cost
profit_day = orders @ margin                                # штук × прибыль с штуки, сложить по изделиям
best_product = products[(orders.sum(axis=0) * margin).argmax()]   # прибыль за неделю по изделиям

In [ ]:
# ✅ Проверка B2 — просто запустите ячейку
try:
    assert np.allclose(unit_cost, [165, 240, 185, 122.5]), f"unit_cost: {unit_cost}"
    assert np.allclose(margin, [285, 160, 165, 127.5]), f"margin: {margin}"
    assert np.allclose(profit_day, [26725, 24707.5, 29077.5, 30555, 37625, 42775]), f"profit_day: {profit_day}"
    assert best_product == "хлеб", f"best_product: {best_product!r} — не путайте прибыль с одного изделия и за неделю"
    print("✅ B2: всё верно!")
except AssertionError as e:
    print("❌ B2: пока не так —", e)
except Exception as e:
    print(f"❌ B2: ошибка {type(e).__name__}: {e}")

✅ B2: всё верно!


## B3 ★ 🔍 Найди ошибку · 🖥 слайды 16–17 · 📓 шаги L1–L2
Коллега посчитал расход так и получил ошибку. Почините: `need_fixed` должен совпасть с `need` из B1.

In [ ]:
try:
    need_fixed = orders @ recipe
except ValueError as e:
    print("ValueError:", e)

In [ ]:
# ✅ Проверка B3 — просто запустите ячейку
try:
    assert np.array_equal(need_fixed, orders @ recipe), "нужно «умножить и сложить» — какой это оператор?"
    print("✅ B3: всё верно!")
except AssertionError as e:
    print("❌ B3: пока не так —", e)
except Exception as e:
    print(f"❌ B3: ошибка {type(e).__name__}: {e}")

✅ B3: всё верно!


🗣 **Почему так?** Другой коллега «починил» иначе: `recipe @ orders.T` — и ошибки нет, форма `(4, 6)`. Но совпадает ли это с `need.T`? Проверьте одной строкой. Почему квадратная матрица `recipe` опасна?

**Ваш ответ:** * умножает поэлементно, а нужно «умножить и сложить», то есть @. Вариант recipe @ orders.T не падает, потому что формы (4, 4) @ (4, 6) подходят, но результат не совпадает с need.T: порядок множителей перепутан, и смысл осей получается неправильный. Квадратная матрица recipe опасна тем, что с ней любой порядок проходит без ошибки, и неправильный результат легко не заметить.

In [ ]:
np.allclose(recipe @ orders.T, need.T)

False

## B4 ★★ Цены поставщика · 🖥 слайд 18 · 📓 шаг L3
Прайс-листа нет, есть три накладные (кг муки, кг масла, кг сахара) и итог каждой в ₸. `kg_prices` — цена килограмма каждого продукта. Проверьте ответ подстановкой.

In [ ]:
Q = np.array([[10, 2, 3],
              [20, 1, 5],
              [ 5, 4, 2]])
totals = np.array([11000, 11000, 16250])
kg_prices = np.linalg.solve(Q, totals)    # [250, 3500, 500]
np.allclose(Q @ kg_prices, totals)        # проверка подстановкой: True

True

In [ ]:
# ✅ Проверка B4 — просто запустите ячейку
try:
    assert np.allclose(Q @ kg_prices, totals), "подстановка не сходится: Q @ kg_prices должно дать totals"
    assert np.allclose(kg_prices, [250, 3500, 500]), f"kg_prices: {kg_prices}"
    print("✅ B4: всё верно!")
except AssertionError as e:
    print("❌ B4: пока не так —", e)
except Exception as e:
    print(f"❌ B4: ошибка {type(e).__name__}: {e}")

✅ B4: всё верно!


### B4.2 ★ 🧱 Вырожденный случай
У второго поставщика тоже три накладные. `det_q2` — определитель; `q2_solution` — решение, а если его нельзя найти — `None` (поймайте `np.linalg.LinAlgError`).

In [ ]:
Q2 = np.array([[10, 2, 3],
               [20, 4, 6],
               [ 5, 4, 2]])
totals2 = np.array([11000, 22000, 16250])
det_q2 = np.linalg.det(Q2)
try:
    q2_solution = np.linalg.solve(Q2, totals2)
except np.linalg.LinAlgError:
    q2_solution = None

In [ ]:
# ✅ Проверка B4.2 — просто запустите ячейку
try:
    assert np.isclose(det_q2, 0), f"det_q2: {det_q2}"
    assert q2_solution is None, "решение не единственное — должно быть None"
    print("✅ B4.2: всё верно!")
except AssertionError as e:
    print("❌ B4.2: пока не так —", e)
except Exception as e:
    print(f"❌ B4.2: ошибка {type(e).__name__}: {e}")

✅ B4.2: всё верно!


🗣 **Почему так?** Посмотрите на первые две накладные `Q2`. Почему по ним нельзя узнать цены?

**Ваш ответ:** вторая накладная — это первая, умноженная на 2 (и количества, и итог). Новой информации она не даёт: фактически у нас 2 уравнения на 3 неизвестных, поэтому цены однозначно не найти, и определитель равен 0.

## B5 ★★ Доля рынка через годы · 🖥 слайды 19–20 · 📓 шаги L5–L6
В районе три пекарни, наша — первая (индекс 0). Матрица переходов покупателей за месяц `P3`: столбец — откуда, строка — куда.
1. `share_100` — доли покупателей через 100 месяцев, если сейчас поровну (`matrix_power`).
2. `stationary` — равновесие через собственный вектор для λ = 1 (вещественные числа, сумма 1).
3. `ours_long` — наша доля в долгую.

In [ ]:
P3 = np.array([[0.8, 0.1, 0.2],
               [0.1, 0.7, 0.2],
               [0.1, 0.2, 0.6]])
share_100 = np.linalg.matrix_power(P3, 100) @ np.full(3, 1/3)
w, V = np.linalg.eig(P3)
k = np.argmin(np.abs(w - 1))
stationary = V[:, k].real / V[:, k].real.sum()
ours_long = stationary[0]

In [ ]:
# ✅ Проверка B5 — просто запустите ячейку
try:
    assert np.allclose(P3.sum(axis=0), 1), "P3 не меняйте"
    assert np.allclose(share_100, [0.4211, 0.3158, 0.2632], atol=1e-4), f"share_100: {share_100}"
    assert not np.iscomplexobj(stationary), "возьмите .real"
    assert np.isclose(stationary.sum(), 1), "доли в сумме — 1: разделите на сумму"
    assert np.allclose(stationary, share_100, atol=1e-6), f"stationary: {stationary}"
    assert np.isclose(ours_long, 8 / 19), f"ours_long: {ours_long}"
    print("✅ B5: всё верно!")
except AssertionError as e:
    print("❌ B5: пока не так —", e)
except Exception as e:
    print(f"❌ B5: ошибка {type(e).__name__}: {e}")

✅ B5: всё верно!


🗣 **Почему так?** Почему равновесие — это собственный вектор именно для собственного значения **1**? (Подсказка: что значит «доли перестали меняться» в виде формулы с `P3`?)

**Ваш ответ:** «доли перестали меняться» значит P3 @ x = x. Это формула собственного вектора M @ v = λ·v при λ = 1: матрица переводит вектор сам в себя.

## B6 ★ Спрос на круассаны · 🖥 слайды 21–23 · 📓 шаги R1–R3
Спрос на круассаны — случайный, в среднем 24 в день (распределение Пуассона). Пекарня печёт по **30** штук.
1. `demand` — спрос за 30 дней: генератор `np.random.default_rng(2026)`, один вызов `poisson`.
2. `days_short` — в скольких днях спрос был больше 30.
3. `lost` — сколько круассанов всего не хватило за месяц. Подсказка: «нехватка, но не меньше нуля» — шаг 3 тетради.

In [ ]:
demand = np.random.default_rng(2026).poisson(24, size=30)
days_short = (demand > 30).sum()
lost = np.maximum(demand - 30, 0).sum()     # нехватка, но не меньше нуля

In [ ]:
# ✅ Проверка B6 — просто запустите ячейку
try:
    assert np.shape(demand) == (30,)
    assert demand[0] == 19 and demand[3] == 32, f"demand[:5] = {demand[:5]} — seed 2026, poisson(24, size=30)?"
    assert days_short == 2, f"days_short: {days_short}"
    assert lost == 3, f"lost: {lost} — отрицательную «нехватку» нужно обнулить"
    print("✅ B6: всё верно!")
except AssertionError as e:
    print("❌ B6: пока не так —", e)
except Exception as e:
    print(f"❌ B6: ошибка {type(e).__name__}: {e}")

✅ B6: всё верно!


## B7 ★★ Монте-Карло: сколько печь? · 🖥 слайд 24 · 📓 шаги R4, 8
Проверим разные объёмы выпечки от 20 до 40 на 10 000 симулированных днях **сразу**, без цикла: столбец спроса `(10000, 1)` против строки запасов `(21,)`.
1. `lost_avg` — средняя нехватка круассанов в день для каждого запаса, форма `(21,)`.
2. `best_stock` — **минимальный** запас, при котором в среднем не хватает меньше 0.5 круассана в день.

In [ ]:
sim = np.random.default_rng(2026).poisson(24, size=(10_000, 1))
stocks = np.arange(20, 41)
lost_avg = np.maximum(sim - stocks, 0).mean(axis=0)   # (10000, 1) - (21,) → (10000, 21), среднее по симуляциям
best_stock = stocks[lost_avg < 0.5][0]                # первый запас, где нехватка < 0.5

In [ ]:
# ✅ Проверка B7 — просто запустите ячейку
try:
    assert np.shape(lost_avg) == (21,), f"lost_avg: форма {np.shape(lost_avg)} — среднее по симуляциям, axis=0"
    assert np.isclose(lost_avg[stocks == 24][0], 1.9528, atol=1e-4), f"при запасе 24: {lost_avg[stocks == 24]}"
    assert (np.diff(lost_avg) <= 0).all(), "чем больше печём, тем меньше нехватка"
    assert best_stock == 29, f"best_stock: {best_stock}"
    print("✅ B7: всё верно!")
except AssertionError as e:
    print("❌ B7: пока не так —", e)
except Exception as e:
    print(f"❌ B7: ошибка {type(e).__name__}: {e}")

✅ B7: всё верно!


### B7.2 ★★★ (по желанию) Не потери, а прибыль
Круассан стоит в производстве 185 ₸, продаётся за 350 ₸. Непроданные — в убыток. `profit_avg` — средняя прибыль в день для каждого запаса (продали `min(спрос, запас)`, испекли `запас`), `best_profit_stock` — запас с максимальной прибылью.
🗣 Почему он **меньше** среднего спроса 24, а не больше?

In [ ]:
profit_avg = ...
best_profit_stock = ...

In [ ]:
# ✅ Проверка B7.2 — просто запустите ячейку
try:
    assert np.shape(profit_avg) == (21,)
    assert best_profit_stock == 23, f"best_profit_stock: {best_profit_stock}"
    assert np.isclose(profit_avg.max(), 3278.7, atol=0.1), f"максимальная прибыль: {profit_avg.max():.1f}"
    print("✅ B7.2: всё верно!")
except AssertionError as e:
    print("❌ B7.2: пока не так —", e)
except Exception as e:
    print(f"❌ B7.2: ошибка {type(e).__name__}: {e}")

❌ B7.2: пока не так — 


**Ваш ответ:** …

## B8 ★ 🔍 Воспроизводимость · 🖥 слайд 21 · 📓 шаг R1
Коллега моделирует спрос за неделю, но все 7 дней получаются одинаковыми. Здесь цикл по дням разрешён.

In [ ]:
def simulate_day_bad(lam):
    rng = np.random.default_rng(42)
    return int(rng.poisson(lam))

print([simulate_day_bad(24) for _ in range(7)])

[28, 28, 28, 28, 28, 28, 28]


Напишите `simulate_day(rng, lam)`, которая **получает генератор снаружи**, и `week_ok` — список из 7 дней, смоделированных одним генератором `np.random.default_rng(42)`.

In [ ]:
def simulate_day(rng, lam):
    return int(rng.poisson(lam))

rng = np.random.default_rng(42)
week_ok = [simulate_day(rng, 24) for _ in range(7)]   # здесь цикл разрешён

In [ ]:
# ✅ Проверка B8 — просто запустите ячейку
try:
    assert len(week_ok) == 7
    assert len(set(week_ok)) > 1, "все дни одинаковые — генератор создаётся заново при каждом вызове?"
    again = np.random.default_rng(42)
    assert week_ok == [simulate_day(again, 24) for _ in range(7)], "с тем же seed результат должен повторяться"
    assert week_ok == [28, 30, 28, 17, 22, 26, 23], f"week_ok: {week_ok} — seed 42?"
    print("✅ B8: всё верно!")
except AssertionError as e:
    print("❌ B8: пока не так —", e)
except Exception as e:
    print(f"❌ B8: ошибка {type(e).__name__}: {e}")

✅ B8: всё верно!


---
# Бонус ★★★: по желанию

## C1 Класс `Bakery` · урок 5 (ООП) + 📓 шаг L2
Соберите расчёты части B в класс:
- `Bakery(products, recipe, price, ing_price)` хранит всё как массивы Numpy. Если число строк `recipe` не равно числу изделий — `ValueError`.
- `need(orders)` — расход ингредиентов по дням;
- `unit_cost()` — себестоимость изделий;
- `profit(orders)` — прибыль по дням (используйте `unit_cost`!).

In [ ]:
class Bakery:
    def __init__(self, products, recipe, price, ing_price):
      self.products = np.array(products)
      self.recipe = np.array(recipe)
      self.price = np.array(price)
      self.ing_price = np.array(ing_price)

      if len(self.recipe) != len(self.products):
        raise ValueError("Строк в рецепте должно быть столько же, сколько изделий")

    def need(self, orders):
      return orders @ self.recipe

    def unit_cost(self):
      return self.recipe @ self.ing_price

    def profit(self, orders):
      margin = self.price - self.unit_cost()
      return orders @ margin

In [ ]:
# ✅ Проверка C1 — просто запустите ячейку
try:
    bk = Bakery(products, recipe, price, ing_price)
    assert np.array_equal(bk.need(orders), orders @ recipe)
    assert np.allclose(bk.unit_cost(), [165, 240, 185, 122.5])
    assert np.allclose(bk.profit(orders)[0], 26725)
    try:
        Bakery(products[:3], recipe, price, ing_price)
        raise AssertionError("3 изделия и 4 строки рецепта — должен быть ValueError")
    except ValueError:
        pass
    print("✅ C1: всё верно!")
except AssertionError as e:
    print("❌ C1: пока не так —", e)
except Exception as e:
    print(f"❌ C1: ошибка {type(e).__name__}: {e}")

✅ C1: всё верно!


## C2 Главное направление данных · 🖥 слайд 19 (разбор) · 📓 шаг L5
Превью метода главных компонент (PCA) из курса ML. Облако из 500 точек вытянуто вдоль какой-то прямой. Её направление — собственный вектор **ковариационной матрицы** с наибольшим собственным значением.
1. `C` — ковариационная матрица `(2, 2)`: `np.cov(pts, rowvar=False)`.
2. `main` — собственный вектор с наибольшим собственным значением; матрица симметричная — возьмите `eigh`. Знак выберите так, чтобы `main[0] > 0`.
3. `slope` — наклон этой прямой: `main[1] / main[0]`.

In [ ]:
rng = np.random.default_rng(4)
x = rng.normal(0, 3, size=500)
y = 0.5 * x + rng.normal(0, 0.5, size=500)
pts = np.column_stack([x, y])

C = ...
main = ...
slope = ...

In [ ]:
# ✅ Проверка C2 — просто запустите ячейку
try:
    assert np.shape(C) == (2, 2) and np.allclose(C, C.T)
    assert np.isclose(np.linalg.norm(main), 1) and main[0] > 0
    assert np.isclose(slope, 0.498, atol=0.002), f"slope: {slope:.3f} — точки строились вокруг y = 0.5x"
    print("✅ C2: всё верно!")
except AssertionError as e:
    print("❌ C2: пока не так —", e)
except Exception as e:
    print(f"❌ C2: ошибка {type(e).__name__}: {e}")

❌ C2: пока не так — 


---
## Перед сдачей
- [ ] «Restart» → «Run All» проходит без ошибок.
- [ ] Все обязательные проверки (A1–A7, B1–B8 вместе с B4.2) дают ✅.
- [ ] Циклов `for` нет, кроме B8.
- [ ] Все «🗣 Ваш ответ» написаны своими словами — хотя бы по одному-двум предложениям.
- [ ] Исходные данные `steps` не изменены (это проверяет A1).